# Single-clip V2V inference (interactive)

Notebook mirror of `inference/v2v_recon.py` — feeds one Waymo clip through Wan 2.1 VACE 1.3B as the control video and writes the reconstructed mp4 plus a side-by-side comparison.

Edit the **Parameters** cell, then run top-to-bottom. Outputs land in `outputs/v2v_recon/<UTC-stamp>_<vid>/` and are previewed inline at the bottom.

**GPU selection.** `GPU_IDS` is a list. The first id is the active device (uses `enable_model_cpu_offload` — peaks ~10-14 GB VRAM, more than fits on one A6000). Extra ids are accepted for forward-compatibility but currently ignored, since 1.3B doesn't need model parallelism. Edit per run, e.g. `[3]`, `[5]`, or `[3, 5]`.

## Parameters

Edit these and re-run. Anything left at `None` falls back to the default in `inference/config.py`.

In [ ]:
# --- Input ---------------------------------------------------------------
VIDEO_ID = "78"                  # short id ('44'), padded ('000044'), or 'waymo_000044'

# --- Prompts -------------------------------------------------------------
PROMPT = None                    # None -> DEFAULT_PROMPT from config
NEGATIVE_PROMPT = None           # None -> DEFAULT_NEGATIVE_PROMPT; '' to disable CFG

# --- Generation ----------------------------------------------------------
HEIGHT = None                    # None -> DEFAULT_HEIGHT (480)
WIDTH = None                     # None -> DEFAULT_WIDTH (832)
NUM_FRAMES = None                # None -> DEFAULT_NUM_FRAMES (49)
STEPS = None                     # None -> DEFAULT_NUM_INFERENCE_STEPS (30)
GUIDANCE = None                  # CFG scale; None -> DEFAULT_GUIDANCE_SCALE (5.0)
CONDITIONING_SCALE = None        # VACE control strength 0-1; None -> 1.0 (full source dominance)
SEED = 0
DTYPE = "bf16"                   # 'bf16' or 'fp16'

# --- Hardware ------------------------------------------------------------
# Edit per run. [3] -> single GPU with cpu_offload; [3, 5] -> model-parallel across two.
GPU_IDS = [3, 5]

# --- Output --------------------------------------------------------------
OUT_FPS = None                   # None -> source fps
WRITE_SIDE_BY_SIDE = True

## Setup

Imports. Helpers (`read_video`, `center_crop_resize`, `write_video`, `write_side_by_side`, `load_pipeline`, `infer_one`, `DTYPES`) live in `inference/utils/`.

In [ ]:
import json
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import torch

# Make the inference/ directory importable regardless of where the kernel was launched.
INFERENCE_DIR = Path('/home/alec/ARV2V/inference')
if str(INFERENCE_DIR) not in sys.path:
    sys.path.insert(0, str(INFERENCE_DIR))

from config import (
    DEFAULT_CONDITIONING_SCALE,
    DEFAULT_GUIDANCE_SCALE,
    DEFAULT_HEIGHT,
    DEFAULT_NEGATIVE_PROMPT,
    DEFAULT_NUM_FRAMES,
    DEFAULT_NUM_INFERENCE_STEPS,
    DEFAULT_PROMPT,
    DEFAULT_WIDTH,
    V2V_RECON_OUTPUTS,
    WAN_VACE_1_3B_LOCAL,
    video_path_for,
)
from utils import (
    DTYPES,
    center_crop_resize,
    infer_one,
    load_pipeline,
    read_video,
    write_side_by_side,
    write_video,
)

## Resolve parameters and validate

Apply config defaults to anything left as `None` and fail fast on missing input/model/GPU.

In [ ]:
params = {
    'video_id': VIDEO_ID,
    'prompt': DEFAULT_PROMPT if PROMPT is None else PROMPT,
    'negative_prompt': DEFAULT_NEGATIVE_PROMPT if NEGATIVE_PROMPT is None else NEGATIVE_PROMPT,
    'height': DEFAULT_HEIGHT if HEIGHT is None else HEIGHT,
    'width': DEFAULT_WIDTH if WIDTH is None else WIDTH,
    'num_frames': DEFAULT_NUM_FRAMES if NUM_FRAMES is None else NUM_FRAMES,
    'steps': DEFAULT_NUM_INFERENCE_STEPS if STEPS is None else STEPS,
    'guidance': DEFAULT_GUIDANCE_SCALE if GUIDANCE is None else GUIDANCE,
    'conditioning_scale': DEFAULT_CONDITIONING_SCALE if CONDITIONING_SCALE is None else CONDITIONING_SCALE,
    'seed': SEED,
    'dtype': DTYPE,
    'gpu_ids': list(GPU_IDS),
    'out_fps': OUT_FPS,
}

src_path = video_path_for(params['video_id'])
model_path = Path(WAN_VACE_1_3B_LOCAL)

for label, p in [('source video', src_path), ('model', model_path)]:
    if not p.exists():
        raise FileNotFoundError(f'{label} not found: {p}')

if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available.')
n_devices = torch.cuda.device_count()
if not params['gpu_ids']:
    raise ValueError('GPU_IDS must contain at least one device index.')
for g in params['gpu_ids']:
    if not 0 <= g < n_devices:
        raise ValueError(f'GPU id {g} out of range (have {n_devices} devices).')
if params['dtype'] not in DTYPES:
    raise ValueError(f"dtype must be one of {list(DTYPES)}")

primary_gpu = params['gpu_ids'][0]
primary_device = torch.device(f'cuda:{primary_gpu}')
print(f'input  : {src_path}')
print(f'model  : {model_path} ({params["dtype"]})')
print(f"gpus   : {params['gpu_ids']} -> primary cuda:{primary_gpu} ({torch.cuda.get_device_name(primary_gpu)})")

## Read and condition the input video

In [ ]:
src_frames, src_fps = read_video(src_path, params['num_frames'])
out_fps = src_fps if params['out_fps'] is None else params['out_fps']
cond_frames = center_crop_resize(src_frames, params['height'], params['width'])
print(f'read   : {len(src_frames)} frames @ {src_fps} fps')
print(f'resize : -> {params["width"]}x{params["height"]}')

## Load the pipeline

Single GPU uses `enable_model_cpu_offload` (the default fast path — components shuttle in/out of VRAM as needed). With multiple GPUs we instead pin the heaviest pieces (transformer, VAE) to the primary device and the text encoder to the second, leaving CPU offload off so nothing thrashes.

In [ ]:
t0 = time.perf_counter()
pipe = load_pipeline(model_path, DTYPES[params['dtype']], gpu_id=primary_gpu)

if len(params['gpu_ids']) > 1:
    print(f"note: VACE-1.3B fits on a single card with cpu_offload (~10-14 GB peak). "
          f"Using cuda:{primary_gpu}; ignoring {params['gpu_ids'][1:]}.")

load_time = time.perf_counter() - t0
print(f'ready  : {load_time:.1f}s   (cpu_offload on cuda:{primary_gpu})')

## Run inference

In [ ]:
print(f"infer  : {params['num_frames']} frames, {params['steps']} steps, "
      f"guidance={params['guidance']}, conditioning_scale={params['conditioning_scale']}, "
      f"seed={params['seed']}")

out_frames, gen_time = infer_one(
    pipe,
    prompt=params['prompt'],
    negative_prompt=params['negative_prompt'],
    cond_frames=cond_frames,
    height=params['height'],
    width=params['width'],
    num_frames=params['num_frames'],
    steps=params['steps'],
    guidance=params['guidance'],
    conditioning_scale=params['conditioning_scale'],
    seed=params['seed'],
    gpu_id=primary_gpu,
)
print(f"done   : {gen_time:.1f}s ({gen_time / params['num_frames']:.2f}s/frame)")

## Save outputs and metadata

In [ ]:
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
run_dir = V2V_RECON_OUTPUTS / f'{stamp}_{src_path.stem}'
run_dir.mkdir(parents=True, exist_ok=True)

input_link = run_dir / 'input.mp4'
input_link.unlink(missing_ok=True)
input_link.symlink_to(src_path)

output_path = run_dir / 'output.mp4'
write_video(out_frames, output_path, fps=out_fps)

sbs_path = None
if WRITE_SIDE_BY_SIDE:
    sbs_path = run_dir / 'side_by_side.mp4'
    write_side_by_side(cond_frames, out_frames, sbs_path, fps=out_fps)

meta = {
    'input': str(src_path),
    'input_fps': src_fps,
    'output_fps': out_fps,
    'model_path': str(model_path),
    **{k: params[k] for k in (
        'prompt', 'negative_prompt', 'conditioning_scale', 'height', 'width',
        'num_frames', 'steps', 'guidance', 'seed', 'dtype', 'gpu_ids',
    )},
    'load_time_s': round(load_time, 2),
    'gen_time_s': round(gen_time, 2),
    'timestamp_utc': stamp,
    'outputs': {
        'input_link': str(input_link),
        'output': str(output_path),
        'side_by_side': str(sbs_path) if sbs_path else None,
    },
}
(run_dir / 'meta.json').write_text(json.dumps(meta, indent=2))
print(f'wrote -> {run_dir}')

## Preview

In [ ]:
from IPython.display import Video, display

preview = sbs_path if sbs_path else output_path
print(preview)
display(Video(str(preview), embed=True))